<a href="https://colab.research.google.com/github/ihanprasetyo/mimic-los-prediction-tutorial/blob/main/mimic_los_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Hospital and ICU Length of Stay using MIMIC-III
AI in Healthcare Tutorial

This tutorial demonstrates how machine learning can predict hospital
resource utilization using the MIMIC-III clinical dataset.

We will predict:

1. Hospital length of stay
2. ICU length of stay

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

## 2. Mount Google Drive and Define Data Path

In [3]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/mimic_data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Load MIMIC Tables

In [5]:
patients = pd.read_csv(DATA_PATH + "PATIENTS.csv")
admissions = pd.read_csv(DATA_PATH + "ADMISSIONS.csv")
icustays = pd.read_csv(DATA_PATH + "ICUSTAYS.csv")

print("Patients:", patients.shape)
print("Admissions:", admissions.shape)
print("ICU stays:", icustays.shape)

Patients: (46520, 8)
Admissions: (58976, 19)
ICU stays: (61532, 12)


## 4. Merge Tables

In [6]:
# Merge PATIENTS and ADMISSIONS
df = pd.merge(admissions, patients, on="SUBJECT_ID", how="inner")

# Merge with ICU stays
df = pd.merge(df, icustays, on=["SUBJECT_ID", "HADM_ID"], how="inner")

print("Merged dataset shape:", df.shape)
df.head()

Merged dataset shape: (61532, 36)


,ROW_ID_x,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,...,ROW_ID,ICUSTAY_ID,DBSOURCE,FIRST_CAREUNIT,LAST_CAREUNIT,FIRST_WARDID,LAST_WARDID,INTIME,OUTTIME,LOS
0,21,22,165315,2196-04-09 12:26:00,2196-04-10 15:54:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,DISC-TRAN CANCER/CHLDRN H,Private,...,22,204798,carevue,MICU,MICU,52,52,2196-04-09 12:27:00,2196-04-10 15:54:00,1.1438
1,22,23,152223,2153-09-03 07:15:00,2153-09-08 19:10:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,...,23,227807,carevue,CSRU,CSRU,14,14,2153-09-03 09:38:55,2153-09-04 15:59:11,1.2641
2,23,23,124321,2157-10-18 19:34:00,2157-10-25 14:00:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME HEALTH CARE,Medicare,...,24,234044,metavision,SICU,SICU,57,57,2157-10-21 11:40:38,2157-10-22 16:08:48,1.1862
3,24,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,...,25,262236,carevue,CCU,CCU,7,7,2139-06-06 16:15:36,2139-06-07 04:33:25,0.5124
4,25,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,...,26,203487,carevue,CCU,CCU,7,7,2160-11-02 03:16:23,2160-11-05 16:23:27,3.5466


## 5. Convert Date Columns

In [7]:
date_cols = [
    "ADMITTIME",
    "DISCHTIME",
    "DOB",
    "INTIME",
    "OUTTIME"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

df[date_cols].head()

,ADMITTIME,DISCHTIME,DOB,INTIME,OUTTIME
0,2196-04-09 12:26:00,2196-04-10 15:54:00,2131-05-07,2196-04-09 12:27:00,2196-04-10 15:54:00
1,2153-09-03 07:15:00,2153-09-08 19:10:00,2082-07-17,2153-09-03 09:38:55,2153-09-04 15:59:11
2,2157-10-18 19:34:00,2157-10-25 14:00:00,2082-07-17,2157-10-21 11:40:38,2157-10-22 16:08:48
3,2139-06-06 16:14:00,2139-06-09 12:48:00,2100-05-31,2139-06-06 16:15:36,2139-06-07 04:33:25
4,2160-11-02 02:06:00,2160-11-05 14:55:00,2101-11-21,2160-11-02 03:16:23,2160-11-05 16:23:27


## 6. Create Length of Stay Variables

In [8]:
# Hospital Length of Stay (days)
df["hospital_los_days"] = (df["DISCHTIME"] - df["ADMITTIME"]).dt.total_seconds() / (24 * 3600)

# ICU Length of Stay (days)
df["icu_los_days"] = (df["OUTTIME"] - df["INTIME"]).dt.total_seconds() / (24 * 3600)

df[["hospital_los_days", "icu_los_days"]].describe()

,hospital_los_days,icu_los_days
count,61532.000000,61522.000000
mean,11.311092,4.917971
std,14.278520,9.638784
min,-0.945139,0.000139
25%,3.907639,1.108070
50%,6.945833,2.092234
75%,13.051562,4.483157
max,294.660417,173.072512


## 7. Compute Patient's Age

In [11]:
df["age"] = df["ADMITTIME"].dt.year - df["DOB"].dt.year

# MIMIC caps age > 89 for privacy
df.loc[df["age"] > 90, "age"] = 90
df.loc[df["age"] < 0, "age"] = np.nan

df["age"].describe()

,age
count,61532.000000
mean,55.573198
std,26.926056
min,0.000000
25%,44.000000
50%,62.000000
75%,76.000000
max,90.000000
